In [1]:
import os
import copy
import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path

from src import utils, stats_utils
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_context("paper", font_scale=1.25)

from src.data.lfs import EulfsDs

%load_ext autoreload
%autoreload 2

In [2]:
config_paths = "paths_config.yml"
config_data = "data_config.yml"

In [3]:
eulfs = EulfsDs(fn_config_path=config_paths, fn_config_data=config_data)

TypeError: calc_gbn_shares_skill_based() missing 1 required positional argument: 'skills_metadata'

In [ ]:
fpath_fr = r"C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles\FR_YEAR_1998_onwards\FR2019_y.csv"
fpath_de = r"C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles\DE_YEAR_1998_onwards\DE2019_y.csv"
fpath_pl = r"C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles\PL_YEAR_1998_onwards\PL2019_y.csv"
fpath_bg = r"C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles\BG_YEAR_1998_onwards\BG2019_y.csv"


In [ ]:
cntry = "MT"
fpath = r"C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles" \
        r"\{}_YEAR_1998_onwards\{}2019_y.csv".format(cntry, cntry)

df = pd.read_csv(fpath, usecols=eulfs.variables, na_values=eulfs.na_values,
                 dtype=eulfs.dtypes_in)
df[df.COUNTRYW == '000-OWN COUNTRY']
df

In [ ]:
df_all = eulfs.read(year=2019, covariates=True)

In [ ]:
col = "isco_label_2"
df_all.dropna(subset=[col])

In [ ]:
def perc_missing(df):
    return df.isna().sum() / df.shape[0]

perc_missing(df_all)

In [ ]:
countries = ["NL"]
perc_missing(df_all.loc[df_all["COUNTRY"].isin(countries)])

In [ ]:
ix_isco = df_all.loc[:, "ISCO3D"].sort_values().index.values
df_all.iloc[ix_isco]

In [ ]:
cols = ["COEFF_share_green_esco_mean", "COEFF_share_brown_esco_mean",
        "COEFF_share_neutral_esco_mean"]

np.allclose(df_all.loc[:, cols].sum(axis=1), df_all.COEFF)
error = np.abs(df_all.loc[:, cols].sum(axis=1) - df_all.COEFF)

df_all.iloc[error.sort_values(ascending=False).index.values]

Group by LFS variables

In [ ]:
age_by_gbn_shares = df_all.groupby("AGE")["COEFF_share_green_esco_mean",
                           "COEFF_share_brown_esco_mean",
                      "COEFF_share_neutral_esco_mean"].sum()

coeff_by_age = df_all.groupby("AGE")["COEFF"].sum()

In [ ]:
age_by_gbn_shares_rel = age_by_gbn_shares.copy()
for col in age_by_gbn_shares_rel.columns:
    age_by_gbn_shares_rel[col] = age_by_gbn_shares_rel[col] / coeff_by_age

In [ ]:
age_by_gbn_shares_rel

In [ ]:
age_by_gbn_shares_rel.plot.bar(stacked=True, figsize=(10, 5))